## Concept 4 — MessagesPlaceholder & Conversation History

### What is it?
`MessagesPlaceholder` reserves a slot in your prompt template where a dynamic list of past messages
is injected at runtime. This is how multi-turn conversations are built.

### Pipeline
```
[System] + [past messages injected here] + [new HumanMessage] → LLM → AIMessage
                ↑
           MessagesPlaceholder
```

### Limitation (what Concept 5 fixes)
LLM returns unstructured text — hard to parse programmatically.
→ Concept 5 forces the LLM to return a Pydantic schema or JSON.

In [1]:
print('Hi')

Hi


In [2]:
import sys
sys.path.insert(0, '.')

from PromptUtils import get_llm
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

llm = get_llm()
print('LLM ready')

LLM ready


### Step 1 — Why History Matters
Without history the LLM forgets everything between turns.

In [3]:
# No history — LLM treats each call independently
no_history_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    ('human',  '{input}'),
])
chain = no_history_prompt | llm | StrOutputParser()

r1 = chain.invoke({'input': 'My name is Mohan.'})
print(f'Turn 1: {r1}')

r2 = chain.invoke({'input': 'What is my name?'})
print(f'Turn 2: {r2}')  # Forgets — has no idea

Turn 1: Hello, Mohan! How can I assist you today?
Turn 2: I'm sorry, but I don't have access to personal information about you unless you've shared it with me in this conversation. How can I assist you today?


### Step 2 — MessagesPlaceholder
Reserve a slot for history. Inject past messages as a list at runtime.

In [4]:
history_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    MessagesPlaceholder(variable_name='history'),  # slot for past messages
    ('human', '{input}'),
])

# Inspect what the prompt looks like with history injected
sample_history = [
    HumanMessage(content='My name is Mohan.'),
    AIMessage(content='Hello Mohan! How can I help you?'),
]

msgs = history_prompt.format_messages(history=sample_history, input='What is my name?')
print(f'{len(msgs)} messages in prompt:')
for m in msgs:
    print(f'  [{m.type}] {m.content}')

4 messages in prompt:
  [system] You are a helpful assistant.
  [human] My name is Mohan.
  [ai] Hello Mohan! How can I help you?
  [human] What is my name?


### Step 3 — Full Conversation Loop (Full History)
Every turn appends to `history`. The LLM sees the entire conversation.

In [5]:
history_chain = history_prompt | llm | StrOutputParser()
history = []

def chat(user_input):
    response = history_chain.invoke({'history': history, 'input': user_input})
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response))
    return response

# Simulate a multi-turn conversation
turns = [
    'My name is Mohan and I am a data scientist.',
    'What is my job?',
    'Can you summarize what you know about me?',
]

for turn in turns:
    print(f'Human: {turn}')
    print(f'AI:    {chat(turn)}')
    print(f'History length: {len(history)} messages\n')

Human: My name is Mohan and I am a data scientist.
AI:    Hello, Mohan! It's great to meet you. As a data scientist, you must be working with a variety of data analysis techniques, machine learning algorithms, and possibly big data technologies. How can I assist you today? Are you looking for information on a specific topic, help with a project, or something else?
History length: 2 messages

Human: What is my job?
AI:    As a data scientist, your job typically involves analyzing and interpreting complex data to help organizations make informed decisions. Your responsibilities may include:

1. **Data Collection and Cleaning**: Gathering data from various sources and ensuring it is clean and usable.

2. **Data Analysis**: Using statistical methods and tools to analyze data sets and extract meaningful insights.

3. **Model Development**: Building predictive models using machine learning algorithms to forecast trends or behaviors.

4. **Data Visualization**: Creating visual representations

### Step 4 — Window Memory Strategy
Keep only the last N messages to prevent the context window from growing too large.

In [6]:
WINDOW_SIZE = 4  # keep last 4 messages (2 turns)
window_history = []

def chat_windowed(user_input):
    # Use only last WINDOW_SIZE messages
    recent = window_history[-WINDOW_SIZE:]
    response = history_chain.invoke({'history': recent, 'input': user_input})
    window_history.append(HumanMessage(content=user_input))
    window_history.append(AIMessage(content=response))
    print(f'  [using {len(recent)} of {len(window_history)-2} stored messages]')
    return response

window_turns = [
    'I am learning Python.',
    'I started last month.',
    'I struggle with decorators.',
    'What have I told you about myself?',  # window may drop turn 1
]

for t in window_turns:
    print(f'Human: {t}')
    print(f'AI:    {chat_windowed(t)}')
    print()

Human: I am learning Python.
  [using 0 of 0 stored messages]
AI:    That's great! Python is a versatile and powerful programming language that's widely used in many areas, such as web development, data analysis, artificial intelligence, scientific computing, and more. How can I assist you with your learning? Do you have any specific questions or topics you'd like to explore?

Human: I started last month.
  [using 2 of 2 stored messages]
AI:    That's awesome! Starting to learn Python is a great step. Since you've been learning for about a month, you might have covered some basics. Here are a few common topics that beginners often explore:

1. **Basic Syntax**: Understanding how to write and run Python code.
2. **Data Types**: Learning about integers, floats, strings, lists, tuples, and dictionaries.
3. **Control Structures**: Using `if` statements, loops (`for`, `while`), and understanding how to control the flow of your program.
4. **Functions**: Defining and calling functions, under

### Full Function

In [7]:
session_history = []

def conversation(user_input):
    response = history_chain.invoke({'history': session_history, 'input': user_input})
    session_history.append(HumanMessage(content=user_input))
    session_history.append(AIMessage(content=response))
    return response

print('Chat session (type questions below):')
demo = ['My favourite language is Python.', 'What language did I mention?']
for q in demo:
    print(f'Q: {q}')
    print(f'A: {conversation(q)}\n')

Chat session (type questions below):
Q: My favourite language is Python.
A: That's great to hear! Python is a versatile and powerful programming language known for its readability and ease of use. Whether you're working on web development, data analysis, machine learning, automation, or scripting, Python has a rich ecosystem of libraries and frameworks to support your projects. What do you enjoy most about Python, or what kind of projects are you working on?

Q: What language did I mention?
A: You mentioned that your favorite language is Python.

